In [3]:
import os
import cv2
import numpy as np
import pandas as pd

import torch
#import torch.utils.data as data
import pytesseract
import torch.optim as optim
import torch.nn as nn

import torchvision
# import torchvision.transforms as transforms
# from torchvision.models.detection import FasterRCNN
# from torchvision.models.detection.backbone_utils import resnet_fpn_backbone


Define the dataset

In [5]:
class DialDetector:
    def __init__(self, model_path):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=False, num_classes=2)
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def detect_dials(self, image):
        image_tensor = torchvision.transforms.functional.to_tensor(image).to(self.device)

        with torch.no_grad():
            output = self.model([image_tensor])
        boxes = output[0]['boxes'].cpu().numpy().astype(int)
        labels = output[0]['labels'].cpu().numpy()
        return boxes, labels

In [6]:
class DialValueDetector:
    def __init__(self):
        self.tesseract_config = '--psm 10'
    
    def recognize_dial_value(self, dial_image):
        dial_image = cv2.cvtColor(dial_image, cv2.COLOR_BGR2GRAY)
        dial_image = cv2.threshold(dial_image, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
        dial_image = cv2.Canny(dial_image, 100, 200)

        dial_text = pytesseract.image_to_string(dial_image, config=self.tesseract_config)

        dial_value = int(dial_text.strip())
        return dial_value

In [ ]:
current_dir = os.getcwd()
train_dataset_dir = os.path.join(current_dir, 'energy_meter_dataset')

dial_detector = DialDetector('')
dial_value_recognizer = DialValueDetector()
